# Grad-CAM Cache ROI Visualization

This notebook uses the existing raw Grad-CAM cache at `checkpoints/roi_records/ftopt_balsam_p25_f25_gastronet_bs32_suppro_probe_mlp_u0_finetune_final.gradcam_cache.npz` and does **not** recompute Grad-CAMs.

It provides:
- shared controls for ROI threshold and minimum ROI size
- a positive-class preview cell
- a negative-class preview cell
- an export cell that writes `checkpoints/roi_records/rois.json` in the existing ROI JSON format


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image, ImageDraw

try:
    import ipywidgets as widgets
    HAS_IPYWIDGETS = True
except ImportError:
    widgets = None
    HAS_IPYWIDGETS = False

from data import build_dataset_dataframe, build_train_val_dataframes
from roi_guidance import compute_roi_geometry_from_bbox, crop_image_to_roi, save_roi_records_to_json

plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.grid'] = False

REPO_ROOT = Path.cwd().resolve()
CACHE_PATH = (REPO_ROOT / 'checkpoints' / 'roi_records' / 'ftopt_balsam_p25_f25_gastronet_bs32_suppro_probe_mlp_u0_finetune_final.gradcam_cache.npz').resolve()
DATA_DIR = (REPO_ROOT / '..' / 'data' / 'Challenge_train_data').resolve()
OUTPUT_JSON_PATH = (REPO_ROOT / 'checkpoints' / 'roi_records' / 'roisposneg.json').resolve()

DEFAULT_THRESHOLD = 0.60
DEFAULT_MIN_ISLAND_COVERAGE = 0.01
DEFAULT_MAX_EXAMPLES = 5
DEFAULT_MAX_PREVIEW_CROPS = 5
ROI_CONTEXT_SCALE = 1.8
ROI_MIN_CROP_SCALE = 0.30
ROI_MAX_ASPECT_RATIO = 1.5
OVERLAY_ALPHA = 0.80
ROI_STRIP_HEIGHT = 220
EXPORT_LABEL_FILTER = None  # Set to None to export ROIs for every image.

if not CACHE_PATH.exists():
    raise FileNotFoundError(f'Grad-CAM cache not found: {CACHE_PATH}')
if not DATA_DIR.exists():
    raise FileNotFoundError(f'Dataset directory not found: {DATA_DIR}')

print(f'Using cache: {CACHE_PATH}')
print(f'Using data dir: {DATA_DIR}')
print(f'ROI JSON output: {OUTPUT_JSON_PATH}')


In [ ]:
def normalize_gradcam_array(raw_cam, eps=1e-8):
    raw_cam = np.asarray(raw_cam, dtype=np.float32)
    cam = np.maximum(raw_cam, 0.0)
    cam_min = float(cam.min())
    cam_max = float(cam.max())
    if cam_max > cam_min + float(eps):
        return (cam - cam_min) / (cam_max - cam_min)
    return np.zeros_like(cam, dtype=np.float32)


def resolve_image_path(image_path):
    image_path = Path(image_path)
    if image_path.is_absolute():
        return image_path
    return (REPO_ROOT / image_path).resolve()


def resize_cam_to_image(cam, image_rgb):
    cam = np.asarray(cam, dtype=np.float32)
    image_h, image_w = image_rgb.shape[:2]
    if cam.shape == (image_h, image_w):
        return cam
    cam_uint8 = np.clip(np.rint(cam * 255.0), 0, 255).astype(np.uint8)
    resized = Image.fromarray(cam_uint8, mode='L').resize((image_w, image_h), Image.BILINEAR)
    return np.asarray(resized, dtype=np.float32) / 255.0


def load_image_rgb(image_path):
    return np.asarray(Image.open(resolve_image_path(image_path)).convert('RGB'))


def overlay_heatmap(image_rgb, cam, max_alpha=OVERLAY_ALPHA):
    cam = resize_cam_to_image(cam, image_rgb)
    cam = np.clip(cam.astype(np.float32), 0.0, 1.0)
    heatmap_rgb = (plt.get_cmap('inferno')(cam)[..., :3] * 255.0).astype(np.uint8)
    alpha = cam[..., None] * float(max_alpha)
    overlay = (1.0 - alpha) * image_rgb.astype(np.float32) + alpha * heatmap_rgb.astype(np.float32)
    return np.clip(np.rint(overlay), 0, 255).astype(np.uint8)


def overlay_mask(image_rgb, mask, color=(255, 255, 0), alpha=0.35):
    base = image_rgb.astype(np.float32).copy()
    tint = np.asarray(color, dtype=np.float32).reshape(1, 1, 3)
    mask = resize_cam_to_image(mask.astype(np.float32), image_rgb) >= 0.5
    mask = mask.astype(bool)[..., None]
    blended = np.where(mask, (1.0 - alpha) * base + alpha * tint, base)
    return np.clip(np.rint(blended), 0, 255).astype(np.uint8)


def draw_bbox(image_rgb, bbox, color=(0, 255, 128), width=4):
    image = Image.fromarray(image_rgb.copy())
    draw = ImageDraw.Draw(image)
    img_w, img_h = image.size
    x0, y0, x1, y1 = [float(value) for value in bbox]
    left = int(round(x0 * img_w))
    top = int(round(y0 * img_h))
    right = int(round(x1 * img_w))
    bottom = int(round(y1 * img_h))

    for offset in range(width):
        draw.rectangle([left - offset, top - offset, right + offset, bottom + offset], outline=tuple(color))
    return np.asarray(image)


def draw_bboxes(image_rgb, roi_records, color=(0, 255, 128), width=4):
    panel = image_rgb.copy()
    for roi_record in roi_records:
        panel = draw_bbox(panel, roi_record['bbox'], color=color, width=width)
    return panel


def find_connected_components(mask):
    mask = np.asarray(mask, dtype=bool)
    if mask.ndim != 2:
        raise ValueError(f'Expected a 2D mask, got shape {mask.shape}.')
    if not mask.any():
        return []

    height, width = mask.shape
    visited = np.zeros_like(mask, dtype=bool)
    components = []
    neighbor_offsets = [(-1, -1), (-1, 0), (-1, 1), (0, -1), (0, 1), (1, -1), (1, 0), (1, 1)]

    ys, xs = np.nonzero(mask)
    for start_y, start_x in zip(ys.tolist(), xs.tolist()):
        if visited[start_y, start_x]:
            continue

        stack = [(int(start_y), int(start_x))]
        visited[start_y, start_x] = True
        coords = []

        while stack:
            y, x = stack.pop()
            coords.append((y, x))
            for dy, dx in neighbor_offsets:
                ny = y + dy
                nx = x + dx
                if 0 <= ny < height and 0 <= nx < width and mask[ny, nx] and not visited[ny, nx]:
                    visited[ny, nx] = True
                    stack.append((ny, nx))

        components.append(np.asarray(coords, dtype=np.int32))

    return components


def build_multi_island_roi_records_from_cam(cam_like, threshold, score, min_island_coverage):
    cam = normalize_gradcam_array(cam_like)
    active_mask = cam >= float(threshold)
    components = find_connected_components(active_mask)
    if not components:
        return []

    height, width = cam.shape
    image_area = float(height * width)
    island_records = []

    for component in components:
        ys = component[:, 0]
        xs = component[:, 1]
        pixel_area = int(len(component))
        source_bbox = (
            float(xs.min()) / float(width),
            float(ys.min()) / float(height),
            float(xs.max() + 1) / float(width),
            float(ys.max() + 1) / float(height),
        )
        island_values = cam[ys, xs]
        coverage = float(pixel_area / image_area)
        mean_activation = float(island_values.mean())
        peak_activation = float(island_values.max())
        signal_mass = float(coverage * mean_activation)
        if coverage < float(min_island_coverage):
            continue

        roi_geometry = compute_roi_geometry_from_bbox(source_bbox)
        island_records.append({
            **roi_geometry,
            'coverage': coverage,
            'score': float(score),
            'source': 'gradcam',
            'pixel_area': pixel_area,
            'peak_activation': peak_activation,
            'mean_activation': mean_activation,
            'signal_mass': signal_mass,
        })

    island_records.sort(key=lambda record: (-record['pixel_area'], -record['peak_activation'], -record['mean_activation']))
    for island_index, record in enumerate(island_records):
        record['island_index'] = int(island_index)
    return island_records


def build_primary_roi_record(island_records, threshold):
    if not island_records:
        return None
    total_coverage = float(sum(record['coverage'] for record in island_records))
    primary_record = dict(island_records[0])
    primary_record.update({
        'coverage': total_coverage,
        'island_count': int(len(island_records)),
        'roi_islands': island_records,
        'roi_threshold': float(threshold),
    })
    return primary_record


def build_multi_crop_strip(image_rgb, roi_records, max_crops=DEFAULT_MAX_PREVIEW_CROPS, target_height=ROI_STRIP_HEIGHT, gap=8):
    if not roi_records:
        return np.zeros((target_height, target_height, 3), dtype=np.uint8)

    pil_image = Image.fromarray(image_rgb)
    crop_panels = []
    for roi_record in roi_records[:max_crops]:
        crop = np.asarray(
            crop_image_to_roi(
                image=pil_image,
                roi_record=roi_record,
                context_scale=ROI_CONTEXT_SCALE,
                min_crop_scale=ROI_MIN_CROP_SCALE,
                jitter_xy=(0.0, 0.0),
                max_aspect_ratio=ROI_MAX_ASPECT_RATIO,
            )
        )
        scale = float(target_height) / float(max(1, crop.shape[0]))
        target_width = max(1, int(round(crop.shape[1] * scale)))
        resized_crop = np.asarray(Image.fromarray(crop).resize((target_width, target_height), Image.BILINEAR))
        crop_panels.append(resized_crop)

    canvas_width = sum(panel.shape[1] for panel in crop_panels) + gap * max(0, len(crop_panels) - 1)
    canvas = np.zeros((target_height, canvas_width, 3), dtype=np.uint8)
    x_offset = 0
    for panel in crop_panels:
        panel_width = panel.shape[1]
        canvas[:, x_offset:x_offset + panel_width] = panel
        x_offset += panel_width + gap
    return canvas


In [ ]:
full_df, class_names = build_dataset_dataframe(str(DATA_DIR))
train_df, val_df, split_class_names = build_train_val_dataframes(str(DATA_DIR), random_state=42)
if class_names != split_class_names:
    raise ValueError('Class names from the full dataframe and split dataframe do not match.')

train_df = train_df.copy()
val_df = val_df.copy()
train_df['split'] = 'train'
val_df['split'] = 'val'
split_df = pd.concat([train_df, val_df], ignore_index=True)
split_df['img'] = split_df['img'].astype(str)

full_df = full_df.copy()
full_df['img'] = full_df['img'].astype(str)
full_df['split'] = full_df['img'].map(split_df.set_index('img')['split'].to_dict()).fillna('unknown')
full_df['resolved_img'] = full_df['img'].map(lambda p: str(Path(p).resolve()))
full_by_resolved = full_df.set_index('resolved_img')
label_names = {idx: name for idx, name in enumerate(class_names)}

cache_bundle = np.load(CACHE_PATH, allow_pickle=True)
cache_image_paths = [str(path) for path in cache_bundle['image_paths'].tolist()]
raw_cams = cache_bundle['raw_cams']

records = []
missing_paths = []
for idx, image_path in enumerate(cache_image_paths):
    resolved_image_path = str(resolve_image_path(image_path))
    if resolved_image_path not in full_by_resolved.index:
        missing_paths.append(image_path)
        continue
    meta_row = full_by_resolved.loc[resolved_image_path]
    cam = normalize_gradcam_array(raw_cams[idx])
    records.append({
        'img': image_path,
        'resolved_img': resolved_image_path,
        'label': int(meta_row['label']),
        'label_name': label_names[int(meta_row['label'])],
        'center': str(meta_row['center']),
        'split': str(meta_row['split']),
        'cam_peak': float(cam.max()),
        'cam_mean': float(cam.mean()),
        'cache_index': int(idx),
    })

if missing_paths:
    raise ValueError(f'{len(missing_paths)} cache images could not be matched back to the dataset. First example: {missing_paths[0]}')

results_df = pd.DataFrame(records).sort_values(['label', 'cam_peak'], ascending=[True, False]).reset_index(drop=True)
positive_label = next((idx for idx, name in label_names.items() if str(name).lower() == 'neo'), max(label_names))
negative_label = next((idx for idx, name in label_names.items() if str(name).lower() in {'ndbe', 'negative'}), min(label_names))

print(f'Loaded {len(results_df)} cached Grad-CAMs.')
print(f'Class names: {class_names}')
print(f'Positive label: {positive_label} ({label_names[positive_label]})')
print(f'Negative label: {negative_label} ({label_names[negative_label]})')

display(results_df.groupby(['label_name', 'split']).size().rename('count').reset_index())
display(results_df.groupby('label_name')[['cam_peak', 'cam_mean']].describe().round(4))
display(results_df.head(10))


In [ ]:
def get_cam_for_row(row):
    return normalize_gradcam_array(raw_cams[int(row.cache_index)])


def select_preview_rows(df, label, count, sort_mode='highest_peak', seed=42):
    subset = df.loc[df['label'] == int(label)].copy()
    if subset.empty:
        return subset.reset_index(drop=True)

    if sort_mode == 'highest_peak':
        subset = subset.sort_values(['cam_peak', 'cam_mean'], ascending=[False, False])
        return subset.head(int(count)).reset_index(drop=True)
    if sort_mode == 'lowest_peak':
        subset = subset.sort_values(['cam_peak', 'cam_mean'], ascending=[True, True])
        return subset.head(int(count)).reset_index(drop=True)
    if sort_mode == 'random':
        return subset.sample(n=min(int(count), len(subset)), random_state=int(seed)).reset_index(drop=True)
    raise ValueError(f'Unsupported sort_mode: {sort_mode}')


def show_label_preview(label, threshold, min_island_coverage, count=DEFAULT_MAX_EXAMPLES, sort_mode='highest_peak', seed=42):
    sample_df = select_preview_rows(results_df, label=label, count=count, sort_mode=sort_mode, seed=seed)
    if sample_df.empty:
        print('No images matched the current preview settings.')
        return sample_df

    fig, axes = plt.subplots(len(sample_df), 4, figsize=(16, 4 * len(sample_df)))
    axes = np.atleast_2d(axes)

    for row_idx, row in enumerate(sample_df.itertuples(index=False)):
        image_rgb = load_image_rgb(row.img)
        cam = get_cam_for_row(row)
        overlay = overlay_heatmap(image_rgb, cam)
        mask = cam >= float(threshold)
        island_records = build_multi_island_roi_records_from_cam(
            cam,
            threshold=float(threshold),
            score=float(row.cam_peak),
            min_island_coverage=float(min_island_coverage),
        )

        bbox_panel = overlay_mask(image_rgb, mask)
        crop_panel = np.zeros((ROI_STRIP_HEIGHT, ROI_STRIP_HEIGHT, 3), dtype=np.uint8)
        if island_records:
            bbox_panel = draw_bboxes(bbox_panel, island_records)
            crop_panel = build_multi_crop_strip(image_rgb, island_records)

        axes[row_idx, 0].imshow(image_rgb)
        axes[row_idx, 0].set_title(f'Original\n{Path(row.img).name}')
        axes[row_idx, 0].axis('off')

        axes[row_idx, 1].imshow(overlay)
        axes[row_idx, 1].set_title(f'Grad-CAM\npeak={row.cam_peak:.3f}')
        axes[row_idx, 1].axis('off')

        axes[row_idx, 2].imshow(bbox_panel)
        axes[row_idx, 2].set_title(f'th={threshold:.2f}\nislands={len(island_records)}')
        axes[row_idx, 2].axis('off')

        axes[row_idx, 3].imshow(crop_panel)
        axes[row_idx, 3].set_title('ROI crop strip' if island_records else 'No ROIs')
        axes[row_idx, 3].axis('off')

        axes[row_idx, 0].set_ylabel(f'{row.label_name}\n{row.center}\n{row.split}', rotation=0, labelpad=40, va='center')

    plt.tight_layout()
    plt.show()
    return sample_df


def build_roi_records(results_df, threshold, min_island_coverage, label_filter=EXPORT_LABEL_FILTER):
    candidate_df = results_df.copy()
    if label_filter is not None:
        candidate_df = candidate_df.loc[candidate_df['label'] == int(label_filter)].copy()

    roi_records = {}
    coverage_values = []
    island_counts = []

    for row in candidate_df.itertuples(index=False):
        cam = get_cam_for_row(row)
        island_records = build_multi_island_roi_records_from_cam(
            cam,
            threshold=float(threshold),
            score=float(row.cam_peak),
            min_island_coverage=float(min_island_coverage),
        )
        primary_record = build_primary_roi_record(island_records, threshold=float(threshold))
        if primary_record is None:
            continue
        roi_records[str(row.img)] = primary_record
        coverage_values.append(float(primary_record['coverage']))
        island_counts.append(int(primary_record['island_count']))

    summary = {
        'threshold': float(threshold),
        'min_island_coverage': float(min_island_coverage),
        'label_filter': None if label_filter is None else int(label_filter),
        'candidate_images': int(len(candidate_df)),
        'roi_records_total': int(len(roi_records)),
        'roi_islands_total': int(sum(island_counts)),
        'acceptance_rate': float(len(roi_records) / len(candidate_df)) if len(candidate_df) else 0.0,
        'mean_coverage': float(np.mean(coverage_values)) if coverage_values else 0.0,
        'median_coverage': float(np.median(coverage_values)) if coverage_values else 0.0,
        'mean_island_count': float(np.mean(island_counts)) if island_counts else 0.0,
        'median_island_count': float(np.median(island_counts)) if island_counts else 0.0,
    }
    return roi_records, summary


def export_roi_json(threshold=None, min_island_coverage=None, output_json_path=OUTPUT_JSON_PATH, label_filter=EXPORT_LABEL_FILTER):
    if threshold is None and HAS_IPYWIDGETS and 'threshold_slider' in globals():
        threshold = float(threshold_slider.value)
    if min_island_coverage is None and HAS_IPYWIDGETS and 'min_coverage_slider' in globals():
        min_island_coverage = float(min_coverage_slider.value)
    threshold = float(DEFAULT_THRESHOLD if threshold is None else threshold)
    min_island_coverage = float(DEFAULT_MIN_ISLAND_COVERAGE if min_island_coverage is None else min_island_coverage)

    roi_records, summary = build_roi_records(
        results_df=results_df,
        threshold=threshold,
        min_island_coverage=min_island_coverage,
        label_filter=label_filter,
    )

    metadata = {
        'gradcam_cache_path': str(CACHE_PATH),
        'data_dir': str(DATA_DIR),
        'image_count_total': int(len(results_df)),
        'threshold': float(threshold),
        'min_island_coverage': float(min_island_coverage),
        'label_filter': None if label_filter is None else int(label_filter),
        'label_filter_name': None if label_filter is None else label_names[int(label_filter)],
        'roi_record_format': 'center_size_plus_bbox',
        'multi_island_rois': True,
        'roi_context_scale': float(ROI_CONTEXT_SCALE),
        'roi_min_crop_scale': float(ROI_MIN_CROP_SCALE),
        'roi_max_aspect_ratio': float(ROI_MAX_ASPECT_RATIO),
    }
    save_roi_records_to_json(output_json_path, roi_records, metadata=metadata)

    print(f'Saved {len(roi_records)} ROI records to {output_json_path}')
    display(pd.DataFrame([summary]).round(4))
    return roi_records, summary


def current_settings_text():
    if HAS_IPYWIDGETS and 'threshold_slider' in globals():
        return f'threshold={threshold_slider.value:.2f}, min_island_coverage={min_coverage_slider.value:.3f}'
    return f'threshold={DEFAULT_THRESHOLD:.2f}, min_island_coverage={DEFAULT_MIN_ISLAND_COVERAGE:.3f}'


## Shared Controls


In [ ]:
if not HAS_IPYWIDGETS:
    print('ipywidgets is not installed, so the preview cells below will use the default settings.')
else:
    threshold_slider = widgets.FloatSlider(
        value=float(DEFAULT_THRESHOLD),
        min=0.0,
        max=1.0,
        step=0.01,
        description='Threshold:',
        readout_format='.2f',
        continuous_update=False,
        layout=widgets.Layout(width='95%'),
        style={'description_width': 'initial'},
    )
    min_coverage_slider = widgets.FloatSlider(
        value=float(DEFAULT_MIN_ISLAND_COVERAGE),
        min=0.0,
        max=0.10,
        step=0.001,
        description='Min ROI size:',
        readout_format='.3f',
        continuous_update=False,
        layout=widgets.Layout(width='95%'),
        style={'description_width': 'initial'},
    )
    example_count_slider = widgets.IntSlider(
        value=int(DEFAULT_MAX_EXAMPLES),
        min=1,
        max=30,
        step=1,
        description='Examples:',
        continuous_update=False,
        layout=widgets.Layout(width='95%'),
        style={'description_width': 'initial'},
    )
    preview_mode_dropdown = widgets.Dropdown(
        options=[('Highest peak', 'highest_peak'), ('Lowest peak', 'lowest_peak'), ('Random', 'random')],
        value='highest_peak',
        description='Preview set:',
        layout=widgets.Layout(width='95%'),
        style={'description_width': 'initial'},
    )
    random_seed_slider = widgets.IntSlider(
        value=42,
        min=0,
        max=999,
        step=1,
        description='Random seed:',
        continuous_update=False,
        layout=widgets.Layout(width='95%'),
        style={'description_width': 'initial'},
    )

    display(widgets.VBox([
        widgets.HTML('<b>Shared ROI tuning controls</b><br>Use these controls for both the positive and negative preview cells below. Rerun the export cell after adjusting them if you want to save the current configuration.'),
        threshold_slider,
        min_coverage_slider,
        example_count_slider,
        preview_mode_dropdown,
        random_seed_slider,
    ]))


## Positive ROI Preview


In [ ]:
if not HAS_IPYWIDGETS:
    print(f'Using defaults: {current_settings_text()}')
    show_label_preview(positive_label, DEFAULT_THRESHOLD, DEFAULT_MIN_ISLAND_COVERAGE)
else:
    positive_output = widgets.interactive_output(
        lambda threshold, min_island_coverage, count, sort_mode, seed: show_label_preview(
            label=positive_label,
            threshold=threshold,
            min_island_coverage=min_island_coverage,
            count=count,
            sort_mode=sort_mode,
            seed=seed,
        ),
        {
            'threshold': threshold_slider,
            'min_island_coverage': min_coverage_slider,
            'count': example_count_slider,
            'sort_mode': preview_mode_dropdown,
            'seed': random_seed_slider,
        },
    )
    display(positive_output)


## Negative ROI Preview


In [ ]:
if not HAS_IPYWIDGETS:
    print(f'Using defaults: {current_settings_text()}')
    show_label_preview(negative_label, DEFAULT_THRESHOLD, DEFAULT_MIN_ISLAND_COVERAGE)
else:
    negative_output = widgets.interactive_output(
        lambda threshold, min_island_coverage, count, sort_mode, seed: show_label_preview(
            label=negative_label,
            threshold=threshold,
            min_island_coverage=min_island_coverage,
            count=count,
            sort_mode=sort_mode,
            seed=seed,
        ),
        {
            'threshold': threshold_slider,
            'min_island_coverage': min_coverage_slider,
            'count': example_count_slider,
            'sort_mode': preview_mode_dropdown,
            'seed': random_seed_slider,
        },
    )
    display(negative_output)


## Export ROI JSON


In [ ]:
# Run this cell after you are happy with the settings above.
# By default it exports only the positive class. Set EXPORT_LABEL_FILTER = None in the config cell to export every label.

exported_roi_records, exported_roi_summary = export_roi_json()
list(exported_roi_records.items())[:3]


## ROI Statistics Overview

Detailed breakdown of ROI island counts per class (negative / positive) and overall. Re-run this cell after changing the threshold or min ROI size sliders to refresh the numbers.

In [ ]:

# ── ROI Statistics Overview ──────────────────────────────────────────────────
# Reads the current threshold / min-coverage values (from sliders if available)
# and sweeps ALL images to produce a full breakdown.

_thresh  = float(threshold_slider.value)    if (HAS_IPYWIDGETS and 'threshold_slider'   in globals()) else DEFAULT_THRESHOLD
_min_cov = float(min_coverage_slider.value) if (HAS_IPYWIDGETS and 'min_coverage_slider' in globals()) else DEFAULT_MIN_ISLAND_COVERAGE

print(f"Computing ROI statistics for ALL images  "
      f"(threshold={_thresh:.2f}, min_island_coverage={_min_cov:.3f}) …")

# ── 1. Sweep every image ─────────────────────────────────────────────────────
_stat_rows = []
for _row in results_df.itertuples(index=False):
    _cam     = normalize_gradcam_array(raw_cams[int(_row.cache_index)])
    _islands = build_multi_island_roi_records_from_cam(
        _cam,
        threshold=_thresh,
        score=float(_row.cam_peak),
        min_island_coverage=_min_cov,
    )
    _n = len(_islands)
    _stat_rows.append({
        'img'            : _row.img,
        'label'          : _row.label,
        'label_name'     : _row.label_name,
        'split'          : _row.split,
        'center'         : _row.center,
        'cam_peak'       : float(_row.cam_peak),
        'cam_mean'       : float(_row.cam_mean),
        'n_islands'      : _n,
        'has_roi'        : _n > 0,
        'total_coverage' : float(sum(r['coverage']        for r in _islands)) if _islands else 0.0,
        'mean_activation': float(np.mean([r['mean_activation'] for r in _islands])) if _islands else 0.0,
        'peak_activation': float(max(r['peak_activation']  for r in _islands)) if _islands else 0.0,
        'mean_signal_mass': float(np.mean([r['signal_mass'] for r in _islands])) if _islands else 0.0,
    })

_stats_df = pd.DataFrame(_stat_rows)

# ── 2. Per-group summary table ───────────────────────────────────────────────
def _group_summary(gdf, name):
    total   = len(gdf)
    has_roi = int(gdf['has_roi'].sum())
    no_roi  = total - has_roi
    n1      = int((gdf['n_islands'] == 1).sum())
    n2      = int((gdf['n_islands'] == 2).sum())
    n3p     = int((gdf['n_islands'] >= 3).sum())
    tot_isl = int(gdf['n_islands'].sum())
    max_isl = int(gdf['n_islands'].max())

    _with_roi = gdf[gdf['has_roi']]
    avg_all   = float(gdf['n_islands'].mean())
    avg_roi   = float(_with_roi['n_islands'].mean())  if has_roi else 0.0
    med_roi   = float(_with_roi['n_islands'].median()) if has_roi else 0.0
    mean_cov  = float(_with_roi['total_coverage'].mean())    if has_roi else 0.0
    med_cov   = float(_with_roi['total_coverage'].median())  if has_roi else 0.0
    mean_peak = float(_with_roi['peak_activation'].mean())   if has_roi else 0.0
    mean_mass = float(_with_roi['mean_signal_mass'].mean())  if has_roi else 0.0

    return {
        'Class'                  : name,
        'Total images'           : total,
        'Images w/ ≥1 ROI'       : has_roi,
        'ROI coverage %'         : f"{100*has_roi/total:.1f}%",
        'Images w/ 0 ROI'        : no_roi,
        'No-ROI %'               : f"{100*no_roi/total:.1f}%",
        'Images w/ 1 ROI'        : n1,
        'Images w/ 2 ROIs'       : n2,
        'Images w/ 3+ ROIs'      : n3p,
        'Total ROI islands'      : tot_isl,
        'Max islands (1 image)'  : max_isl,
        'Avg islands / image'    : round(avg_all, 3),
        'Avg islands / ROI image': round(avg_roi, 3),
        'Median islands / ROI'   : round(med_roi, 1),
        'Mean coverage (ROI img)': round(mean_cov, 4),
        'Median coverage'        : round(med_cov, 4),
        'Mean peak activation'   : round(mean_peak, 4),
        'Mean signal mass'       : round(mean_mass, 6),
    }

_neg_df  = _stats_df[_stats_df['label'] == negative_label]
_pos_df  = _stats_df[_stats_df['label'] == positive_label]

_summary_rows = [
    _group_summary(_neg_df,    label_names[negative_label]),
    _group_summary(_pos_df,    label_names[positive_label]),
    _group_summary(_stats_df,  'TOTAL'),
]
_summary_table = pd.DataFrame(_summary_rows).set_index('Class').T

print(f"\n{'='*72}")
print(f"  ROI STATISTICS  |  threshold={_thresh:.2f}  min_coverage={_min_cov:.3f}")
print(f"{'='*72}")
display(_summary_table)

# ── 3. Per-split × class breakdown ───────────────────────────────────────────
print("\n── Per split × class ──────────────────────────────────────────────────")
_split_agg = (
    _stats_df
    .groupby(['label_name', 'split'])
    .agg(
        total_images  =('n_islands', 'count'),
        with_roi      =('has_roi',   'sum'),
        no_roi        =('has_roi',   lambda x: (~x).sum()),
        total_islands =('n_islands', 'sum'),
        avg_islands   =('n_islands', 'mean'),
        median_islands=('n_islands', 'median'),
        max_islands   =('n_islands', 'max'),
    )
    .round(3)
)
_split_agg['roi_%'] = (_split_agg['with_roi'] / _split_agg['total_images'] * 100).round(1)
display(_split_agg)

# ── 4. Island-count frequency tables ─────────────────────────────────────────
print("\n── Island-count frequency (how many images have exactly N islands) ────")
_freq_rows = []
_max_n = int(_stats_df['n_islands'].max())
_cap   = min(_max_n, 6)
for _n in range(0, _cap + 1):
    _label = f"{'≥' if _n == _cap and _cap < _max_n else ''}{_n} islands"
    _row   = {'N islands': _label}
    for _gname, _gdf in [(label_names[negative_label], _neg_df),
                          (label_names[positive_label], _pos_df),
                          ('TOTAL', _stats_df)]:
        _cnt = int((_gdf['n_islands'] >= _n if (_n == _cap and _cap < _max_n)
                    else _gdf['n_islands'] == _n).sum())
        _row[_gname]           = _cnt
        _row[f'{_gname} (%)']  = f"{100*_cnt/len(_gdf):.1f}%"
    _freq_rows.append(_row)
display(pd.DataFrame(_freq_rows).set_index('N islands'))

# ── 5. Distribution bar chart ────────────────────────────────────────────────
_fig, _axes = plt.subplots(1, 3, figsize=(16, 4))
_colors = {'neg': '#4878D0', 'pos': '#EE854A', 'total': '#6ACC65'}

for _ax, (_gname, _gdf, _col) in zip(_axes, [
    (label_names[negative_label], _neg_df,   _colors['neg']),
    (label_names[positive_label], _pos_df,   _colors['pos']),
    ('Total',                     _stats_df, _colors['total']),
]):
    _vc   = _gdf['n_islands'].value_counts().sort_index()
    _xmax = min(int(_vc.index.max()), 6)
    _xs   = list(range(0, _xmax + 1))
    _ys   = [int((_gdf['n_islands'] >= _x if _x == _xmax and _xmax < int(_gdf['n_islands'].max()) else _gdf['n_islands'] == _x).sum()) for _x in _xs]
    _bars = _ax.bar(_xs, _ys, color=_col, edgecolor='white', linewidth=0.6)
    for _bar, _y in zip(_bars, _ys):
        if _y > 0:
            _ax.text(_bar.get_x() + _bar.get_width()/2, _bar.get_height() + 0.3,
                     str(_y), ha='center', va='bottom', fontsize=8)
    _ax.set_title(f'{_gname}', fontsize=11, fontweight='bold')
    _ax.set_xlabel('ROI islands per image')
    _ax.set_ylabel('Number of images')
    _xlabels = [str(x) if x < _xmax else f'{_xmax}+' for x in _xs]
    _ax.set_xticks(_xs)
    _ax.set_xticklabels(_xlabels)
    _ax.set_ylim(0, max(_ys) * 1.15 if max(_ys) > 0 else 1)

_fig.suptitle(
    f'ROI Island Distribution  (threshold={_thresh:.2f}, min_coverage={_min_cov:.3f})',
    fontsize=12, fontweight='bold', y=1.01,
)
plt.tight_layout()
plt.show()

# ── 6. Cam-peak distribution: images with vs without ROI ─────────────────────
_fig2, _axes2 = plt.subplots(1, 2, figsize=(14, 4))
for _ax, (_gname, _gdf) in zip(_axes2, [
    (label_names[negative_label], _neg_df),
    (label_names[positive_label], _pos_df),
]):
    _ax.hist(_gdf.loc[ _gdf['has_roi'],  'cam_peak'], bins=30, alpha=0.7, label='Has ROI',  color='steelblue')
    _ax.hist(_gdf.loc[~_gdf['has_roi'], 'cam_peak'], bins=30, alpha=0.7, label='No ROI',   color='tomato')
    _ax.set_title(f'{_gname}: cam_peak distribution')
    _ax.set_xlabel('cam_peak (normalised GradCAM max)')
    _ax.set_ylabel('Images')
    _ax.legend()
_fig2.suptitle('GradCAM Peak Activation — images with vs. without detected ROI',
               fontsize=11, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print("\nDone.")
